# Session 4 — Designing an Agent Evaluation Pipeline

> **An evaluator is a hypothesis about a failure: it says where to look and what
> would count as failing. Until something has failed it, you have not measured
> anything — you have decorated.**

Sessions 1–3 got you here:

| | |
|---|---|
| **S1** | An output-only evaluation is only as correct as the model grading it. |
| **S2** | The trace contains what no answer-grader can reach — the path, the waste, the bill. |
| **S3** | The path is a design choice. You pick who decides, then pay for it every run. |
| **S4** | And a check on either one is only a *claim* until something fails it. |

**No `%pip install` today.** You migrated to the repo as homework. We open with `git pull`.

**What you will build, by hand:**
1. One dataset example — including what would count as a correct answer
2. One tool-correctness evaluator
3. **The falsification run** — prove your evaluator can fail
4. One experiment, and one written judgement about a metric

## 0 · Environment

**This notebook is not standalone, and that is deliberate.** It imports `evalkit`,
`seeds` and `eval_dataset` — real files in `course-repo` that you can open, read, diff
and commit. **Open the `course-repo` folder in VS Code and run this notebook from
inside it.** It will not work in Colab: half of today happens in a terminal.

```bash
cd course-repo && git pull
```

If the version assert fails you are on stale code, and every number you produce today
will silently be the previous version's.

In [ ]:
# ---------------------------------------------------------------------------
# BOOTSTRAP. Finds course-repo and puts it on sys.path.
#
# This notebook is NOT standalone. It imports evalkit / seeds / eval_dataset,
# which are real files in course-repo -- that is the point of the migration.
# Run it from inside the repo, in VS Code.
# ---------------------------------------------------------------------------
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
BAR = "=" * 66


def find_repo(start=None, needle="evalkit.py", up=4):
    here = Path(start or os.getcwd()).resolve()
    for d in [here, *here.parents][:up + 1]:
        if (d / needle).exists():
            return d
        if (d / "course-repo" / needle).exists():
            return d / "course-repo"
    return None


def die(*lines):
    print(BAR)
    for ln in lines:
        print(ln)
    print(BAR)
    raise SystemExit(1)


REPO = find_repo()

if REPO is None and IN_COLAB:
    die("You are in Google Colab. Session 4 runs in VS Code, from the repo.",
        "",
        "Half of today happens in a terminal (python test_my_evaluators.py),",
        "and you edit my_evaluators.py in an editor. Colab gives you neither.",
        "",
        "  1. Open VS Code",
        "  2. File > Open Folder > course-repo",
        "  3. Open session4_eval_pipeline.ipynb from INSIDE that folder",
        "  4. Pick the .venv interpreter (bottom-right, or Cmd/Ctrl+Shift+P >",
        "     Python: Select Interpreter)",
        "",
        "Stuck? Pair up. One working machine per pair is enough -- the block",
        "that matters today needs no API keys at all.")

if REPO is None:
    die("Could not find course-repo.",
        "",
        "This notebook is running from:",
        "  " + os.getcwd(),
        "",
        "and there is no evalkit.py here or in the 4 directories above it.",
        "",
        "Move the notebook INTO course-repo, or open that folder in VS Code",
        "and open the notebook from there. Then: git pull")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("course-repo ->", REPO)

missing = [f for f in ("evalkit.py", "seeds.py", "eval_dataset.py",
                       "my_evaluators.py", "test_my_evaluators.py")
           if not (REPO / f).exists()]
if missing:
    die("Found the repo, but these files are missing:",
        "  " + ", ".join(missing),
        "",
        "Run: git pull")

# Fixtures are only needed from Hands-on 3 onward. Warn, do not stop -- there is
# no reason a missing file at minute 3 should cost you the first forty minutes.
if not (REPO / "seed_fixtures.json").exists():
    print()
    print("WARNING: seed_fixtures.json is missing.")
    print("Everything up to Hands-on 3 works without it. Hands-on 3 does not.")
    print("Try: git pull   -- if that does not bring it, ask the instructor.")

In [ ]:
# Gotcha #5: LangSmith caches the project name on FIRST read with an lru_cache.
# A project set after something has already read it lands your traces in
# `default` -- with no error at all. So .env loads before evalkit imports.
try:
    from dotenv import load_dotenv
except ImportError:
    die("pip install python-dotenv",
        "(it is in requirements.txt -- git pull first)")

if not load_dotenv(REPO / ".env"):
    die("No .env found at: " + str(REPO / ".env"),
        "",
        "  cp .env.example .env",
        "",
        "then fill in ANTHROPIC_API_KEY, LANGSMITH_API_KEY, TAVILY_API_KEY")

import evalkit, seeds, eval_dataset

EXPECTED_VERSION = "s4-2026-09-01a"
for mod in (evalkit, seeds, eval_dataset):
    assert mod.__version__ == EXPECTED_VERSION, (
        mod.__name__ + " is " + mod.__version__ + ", expected " + EXPECTED_VERSION
        + ". git pull, then RESTART THE KERNEL -- a stale module cached in a "
        "running kernel reports the previous version's numbers and nothing warns you."
    )

print("resolved LangSmith project ->", evalkit.env_setup())   # must NOT say 'default'

In [ ]:
# One CHAT alias. One variable swaps provider — set COURSE_PROVIDER in your .env
# to `openai` or `google` if that is what you are running on your own credits.
chat = evalkit.get_chat()
print(f"provider={evalkit.PROVIDER}  model={evalkit.MODEL_IDS[evalkit.PROVIDER]}  "
      f"reasoning_effort pinned={evalkit.EFFORT_PINNED}")

## 1 · The number that starts today

Session 3 measured three architectures. Here is the task-success column:

| | react | toolcall | workflow |
|---|---|---|---|
| **task success** | **1.00** | **1.00** | **1.00** |
| easy — tool calls | 1.0 | 1.0 | **3.0** |
| easy — cost USD | 0.0070 | 0.0072 | **0.0127** |

**Same three runs.** One of them costs 1.8× the others and issues three searches
where one would do. The grader said everything was fine.

That grader was not measuring. **It was agreeing.**

> **Discuss with your partner, 60 seconds — one number each:**
> how many of the four evaluators you will write today do you think can catch
> that workflow row? Write the number down before you look.

## 2 · What to evaluate — five types, one object

The syllabus names five kinds of evaluation. They are not five subjects. They are
**five fields of the same run**, and you already know how to read a run.

| type | where it looks | today |
|---|---|---|
| **Outcome** | the last message | you write one |
| **Process / Trajectory** | the ordered tool calls | pre-built, you read it |
| **Tool correctness** | which tool, what arguments | **you write this** |
| **State** | the graph state between nodes | → **Session 11** owns it |
| **Safety** | any of the above, adversarially | → **Session 5** owns it |

Two of those are forward-references, said out loud rather than skipped. State
evaluation needs a stateful workflow to evaluate, and you build one in Session 11.
Safety needs adversarial datasets, which is Session 5's whole content.

In [ ]:
# Run the agent ONCE and look at every field an evaluator could read.
# This is the only live agent run in the first half of the session.
agent = evalkit.build_agent()
result = agent.invoke({"messages": [
    "What is the latest released version of the `langgraph` package on PyPI?"]})

messages   = evalkit.as_messages(result)
answer     = evalkit.final_text(result)
tool_calls = evalkit.tool_calls_from_messages(messages)
evidence   = evalkit.evidence_from_messages(messages)

print("OUTCOME     ", repr(answer[:110]), "...")
print("TRAJECTORY  ", [tc["name"] for tc in tool_calls])
print("TOOL ARGS   ", [tc["args"] for tc in tool_calls])
print("EVIDENCE    ", len(evidence), "tool results,",
      sum(len(e) for e in evidence), "chars")
print("PROCESS     ", len(messages), "messages")

### Gotcha, and it has bitten this course three times

`final_text()` is not `result["messages"]`.

A LangGraph root's `outputs` is the **whole message list**, and it carries
`usage_metadata`. Dump that to text for a grounding check and you will read prompt
token counts as factual claims. Session 2's grounding checker reported `566` as a
hallucinated value. 566 was the input token count of reasoning step 1.

**An evaluator is only as good as the field it reads.**

## 3 · Offline and online — two function calls, not two definitions

| | **OFFLINE** (pre-deployment) | **ONLINE** (post-deployment) |
|---|---|---|
| when | before you ship | on live traffic |
| you have | a dataset **you wrote**, so reference outputs exist | a trace. That is all. |
| evaluator reads | `inputs, outputs, reference_outputs` | `run` |
| the question | did it match what I said was correct? | can I still say something is wrong? |

That second column is not a limitation to apologise for. **It is the definition.**
Online evaluators receive no reference outputs, because in production nobody wrote
down the right answer.

You can still catch a great deal:

In [ ]:
import inspect
print("OFFLINE — reads the dataset's answer key")
print(inspect.getsource(evalkit.tool_correctness))
print("ONLINE  — no reference output exists, only the trace")
print(inspect.getsource(evalkit.online_empty_retrieval))

## 4 · Metrics — the vocabulary, and the trap in each

| metric | what it reads | how it lies to you |
|---|---|---|
| **Task Success Rate** | outcome | passes everything (S3: 1.00 / 1.00 / 1.00) |
| **Tool Selection Accuracy** | tool calls | right tool, wrong *arguments*, still scores 1 |
| **Hallucination Rate** | answer vs evidence | needs a judge; the judge needs its own check |
| **Average Latency** | wall clock | an average hides the tail that pages you at 3am |
| **Token Cost** | trace | not comparable across providers unless you define it |

Session 3's definition, still in force: **`tokens_billed` = output + *uncached* input.**
Cached reads excluded, reasoning tokens in their own column. Say the definition out
loud *before* putting two numbers side by side.

---
# Hands-on 1 · Write one dataset row
### *Syllabus: create evaluation datasets for browser search, information retrieval, multi-hop reasoning and report generation*

**One dataset, four categories — not four datasets.** Filter by category when you
want one slice, and pass the filtered list straight in as `data=`.

Writing the question takes ten seconds. Writing down **what would count as correct**
is where every real disagreement lives — and it is where Session 1 drew blood: the
"newest Claude model" probe had two defensible answers, and a reference that accepted
only one would mark a correct answer *wrong*.

In [ ]:
for cat in eval_dataset.CATEGORIES:
    rows = eval_dataset.local_rows(cat)
    print(f"\n=== {cat}  ({len(rows)} rows) ===")
    for r in rows:
        print(" Q:", r["inputs"]["question"][:78])
        print("   must_contain   ", r["outputs"]["must_contain"])
        print("   expected_tools ", r["outputs"]["expected_tools"],
              " forbidden:", r["outputs"]["forbidden_tools"],
              " budget:", r["outputs"]["max_tool_calls"])

### Your turn

Your pair has been assigned **one category**. Write one row for it.

Rules that make it a real dataset row rather than a question:

- **The answer must be checkable in ten seconds.** If you cannot verify it yourself,
  "is this true?" becomes "I'm not an expert" and the row is worthless.
- `must_contain` — as few keywords as will do the job. Every extra one is another way
  to mark a correct answer wrong.
- `max_tool_calls` — a *fair* budget, not a generous one. A budget nothing exceeds is
  not a budget.
- Leave `expected_trajectory` alone. Pinning an exact tool sequence turns every
  reasonable alternative path into a failure, and then you are measuring conformity.

In [ ]:
# TODO — your row. Replace every TODO.
MY_ROW = {
    "inputs": {"question": "TODO"},
    "outputs": {
        "must_contain":    [],          # TODO: fewest keywords that prove correctness
        "expected_tools":  [],          # TODO: "web_search" and/or "calculator"
        "forbidden_tools": [],          # TODO: which tool would be a mistake here?
        "max_tool_calls":  3,           # TODO: a fair budget
    },
    "metadata": {"category": "TODO",    # TODO: your assigned category
                 "difficulty": "easy",
                 "verify_url": "TODO"}, # TODO: where you checked the answer
}

In [ ]:
# Validate BEFORE uploading. A malformed row fails inside evaluate(), fifteen
# minutes later, on someone else's screen.
def validate(row):
    problems = []
    o, m = row["outputs"], row["metadata"]
    if "TODO" in row["inputs"]["question"]:  problems.append("question is still TODO")
    if not o["must_contain"]:                problems.append("must_contain is empty — nothing can fail")
    if len(o["must_contain"]) > 3:           problems.append("more than 3 keywords — brittle")
    if not o["expected_tools"]:              problems.append("expected_tools is empty")
    if m["category"] not in eval_dataset.CATEGORIES:
        problems.append(f"category must be one of {eval_dataset.CATEGORIES}")
    if not m.get("verify_url") or m["verify_url"] == "TODO":
        problems.append("no verify_url — nobody can re-check this when it goes stale")
    return problems

issues = validate(MY_ROW)
print("\n".join("  - " + p for p in issues) if issues else "row looks well-formed")

---
# Hands-on 2 · Write one evaluator — **in your editor, not in here**
### *Syllabus: Tool Correctness Eval · convert debugging observations into automated evaluators · rule-based automated evaluations*

Open **`my_evaluators.py`** in VS Code. That is where you work for the next ten
minutes. This notebook only *runs* what you write there.

**Why a file and not a cell.** An evaluator in a cell cannot be imported, diffed,
run from a terminal, or committed. An evaluator in a file can do all four. In
production, evaluators are reviewed code that runs in CI — that is Session 12, and
this is where it starts.

**The signature is keyword-matched.** Declare only the parameters you need, from
`inputs`, `outputs`, `reference_outputs`, `run`, `example`. LangSmith fills in what
you asked for and nothing else. Return `{"key", "score", "comment"}` — or a bare
`bool`, in which case the function's name becomes the metric name.

In [ ]:
# One that ships, so you can see the shape. Nothing is hidden — you just will
# not type it.
print(inspect.getsource(evalkit.trajectory_no_waste))

### Your turn — two windows

**Editor:** fill in `my_tool_check` in `my_evaluators.py`. It must decide whether
the agent used the tool this task needed and avoided the one that would be a mistake.

**Terminal:**

```bash
python test_my_evaluators.py
```

Two seconds, no API calls. It will tell you whether your evaluator can fail.

Two hints that are really the whole exercise:

1. `expected_tools` is a **set**, not a sequence. Order is the trajectory
   evaluator's job. Keeping them apart is what stops one metric failing for two
   unrelated reasons — the fastest way to make a number uninterpretable.
2. Decide what you do when the agent calls **no tools at all**. That is a real case
   and your rule will meet it in ten minutes.

---
# Hands-on 3 · Prove it can fail
### *The block this session is built on — you run it in the terminal, we read it here*

You have an evaluator. **You have no evidence it works.**

An evaluator that returns the same verdict on every run has not measured anything.
It cannot distinguish a good run from a bad one, so its score carries no
information — however much you like the number.

This is not hypothetical for this course. Three times now the *harness* has been the
broken thing, not the agent:

- Session 2's classifier tagged its own deliberately-clean control as broken.
- The span-counting walk counted every tool call twice.
- This session's own preflight printed `[GO] healthy passes every evaluator` on
  **zero data**, and told us to retire three working seeds.

**Test on the control first.** `test_my_evaluators.py` does, and stops if it fails.

```bash
python test_my_evaluators.py      # your terminal, two seconds, no API calls
```

In [ ]:
# Four runs, captured live in pre-flight and saved. No API calls, no waiting.
# Iterating on an evaluator against fixtures takes milliseconds; re-invoking the
# agent takes 25 seconds a row.
fixtures = seeds.load_fixtures()
print(f"{len(fixtures)} saved runs:",
      sorted({f['seed'] for f in fixtures}) or "NONE — ask the instructor")

for name in ("healthy", "redundant"):
    f = next(x for x in fixtures if x["seed"] == name)
    print(f"\n--- {name} ---")
    print("  answer :", f["answer"][:88].replace("\n", " "), "...")
    print("  tools  :", [tc["name"] for tc in f["tool_calls"]])
    print("  queries:", [tc["args"].get("query", "") for tc in f["tool_calls"]])

**Look at `redundant` above before you run anything else.**

Correct answer. Right tool. Grounded in real search results. Cites its sources.

And it issued the same query twice.

Nothing that reads the *answer* can see that. Only something that reads the *path*.

### And now the same thing on the projector

You just ran this in your terminal. Here it is again in the notebook, reading the
**same file you edited**, so the whole room is looking at one table.

If the notebook disagrees with your terminal, you edited the file after Python
imported it. `importlib.reload` below fixes that — and the fact that you have to
think about it at all is what living in version control costs you.

In [ ]:
import importlib
import my_evaluators
importlib.reload(my_evaluators)     # picks up your edits without a kernel restart

REF = {"must_contain": ["1.2"],
       "expected_tools": ["web_search"],
       "forbidden_tools": ["calculator", "package_registry"],
       "max_tool_calls": 1}
INPUTS = {"question": "What is the latest released version of `langgraph` on PyPI?"}

# Yours first, then the shipped ones for comparison.
ALL_EVALUATORS = list(my_evaluators.MY_EVALUATORS) + list(evalkit.OFFLINE_EVALUATORS)

rows, grid = [], {}
for f in fixtures:
    res = evalkit.run_offline_evaluators(INPUTS, f, REF, ALL_EVALUATORS)
    rows.extend(res)
    grid.setdefault(f["seed"], {})
    for r in res:
        grid[f["seed"]].setdefault(r["key"], []).append(r["score"])

keys = sorted({k for row in grid.values() for k in row})
order = [s for s in ("healthy", "wrong_tool", "redundant", "empty_search") if s in grid]
print(f"{'seed':<14}" + "".join(f"{k:<22}" for k in keys))
print("-" * (14 + 22 * len(keys)))
for seed in order:
    cells = []
    for k in keys:
        v = [x for x in grid[seed].get(k, []) if x is not None]
        cells.append("PASS" if v and all(v) else "fail" if v else "skip")
    print(f"{seed:<14}" + "".join(f"{c:<22}" for c in cells))

print()
evalkit.print_discrimination(evalkit.discrimination_report(rows))

### Read your matrix

> **With your partner, 3 minutes. Three specific answers:**
>
> 1. Does `healthy` pass everything? If not, **your evaluator is broken, not the
>    agent.** Fix that before anything else.
> 2. Which seed does `my_tool_check` fail? If the answer is *none*, you have no
>    evidence it can fail — go back and fix the predicate.
> 3. Look at the `redundant` row. How many of your three evaluators caught it?
>    **Compare that number to the one you wrote down at the start of the session.**

If an evaluator is ALL-PASS or ALL-FAIL across this matrix, it is decoration.
Retire it or fix it. That is the rule, and it applies to code you like.

---
# Hands-on 4 · Run the experiment
### *Syllabus: configure and run LangSmith experiments using rule-based automated evaluations*

Now against the real dataset, with the real agent. **This is the slow part** — one
agent run per row. Set `max_concurrency` and do not run it twice out of habit.

In [ ]:
from langsmith import Client
client = Client()

dataset, n = eval_dataset.push(client)
print(f"dataset: {dataset.name}  (+{n} new examples)")

# Uncomment to add YOUR row once validate() is clean:
# client.create_examples(dataset_id=dataset.id, examples=[MY_ROW])

In [ ]:
target = evalkit.make_target()      # def target(inputs: dict) -> dict

results = client.evaluate(
    target,
    data=eval_dataset.slice_for(client, "browser_search"),   # one slice, not all 8
    evaluators=ALL_EVALUATORS,
    experiment_prefix="s4-yourname",
    max_concurrency=4,
    num_repetitions=1,          # Session 5 tells you what number belongs here
    metadata={"provider": evalkit.PROVIDER},
)
rows_out = list(results)
print(f"\n{len(rows_out)} rows evaluated — open the link above")

In [ ]:
# Discrimination across the DATASET. Different question from the seed matrix:
# there we asked "can this evaluator tell a broken agent from a healthy one",
# here we ask "does it tell these questions apart".
def _key(r):
    return getattr(r, "key", None) or (r.get("key") if isinstance(r, dict) else None)
def _score(r):
    return getattr(r, "score", None) or (r.get("score") if isinstance(r, dict) else None)

flat = []
for row in rows_out:
    er = row.get("evaluation_results") if isinstance(row, dict) else None
    for r in (er or {}).get("results", []) if isinstance(er, dict) else []:
        flat.append({"key": _key(r), "score": _score(r)})

evalkit.print_discrimination(evalkit.discrimination_report(flat))

### Your turn — one written judgement

> **Name one evaluator on that table that is NOT discriminating, and say in one
> sentence why.** Not "it needs more data" — *why*, mechanically. What is it reading
> that cannot vary across these rows?
>
> This is your homework's first field. Write it now while the table is in front of you.

**Iterating without re-running the agent.** Every re-run above costs one agent
invocation per row. It does not have to:

```python
from langsmith import evaluate_existing
evaluate_existing("s4-yourname-<id>", evaluators=[my_better_evaluator])
```

Same rows, same outputs, new evaluators, **zero agent cost**. That is also the
offline/online distinction in one function call — you are now scoring traces that
already happened.

---
# 5 · The expensive option — LLM-as-a-Judge
### *Syllabus: convert debugging observations into automated evaluators (LLM-as-a-Judge)*

Look back at the `report_gen` rows in the dataset. Their `must_contain` is almost
empty, and that was not laziness.

> **Ten seconds, shout it out: what keyword would prove a comparison of offline and
> online evaluation is any good?**

There isn't one. Some questions cannot be graded by code, and that is the *only*
reason to reach for a model.

In [ ]:
# Twelve lines. Hand-rolled on purpose: it uses the CHAT alias, so it works on
# whichever provider you are running, and it adds nothing to a pin set forty
# people already installed. Session 8 introduces the real package.
print(inspect.getsource(evalkit.make_groundedness_judge))

In [ ]:
judge = evalkit.make_groundedness_judge()

# The empty_search seed: every search came back empty.
empty = next(f for f in fixtures if f["seed"] == "empty_search")
print("evidence retrieved:", empty["evidence"])
print("answer:", empty["answer"][:200])
print("\njudge ->", judge({"question": INPUTS["question"]}, empty))

### Two things measured in pre-flight, and the second one matters more

**1. What it costs.** `$0.0014` and **12.83 seconds** per example.

The cost is nothing. The **latency** is the argument: 12.8s across a 50-row dataset is
eleven minutes of an eval run, against milliseconds for every rule-based check you
wrote today. That is why the judge is the last thing you reach for, not the first.

**2. It disagreed with itself.** Run twice on *identical* empty evidence, the judge
returned `GROUNDED` once and `UNGROUNDED` once.

Sit with that. It is an evaluator, and this session's rule is that an evaluator gets
tested against a known-good run before you trust it. **We have not done that to the
judge today, and you should not treat its score as a fact.**

Rubrics, position bias, verbosity bias, self-preference, and how to check a judge — 
that is **Session 8**, and this is why it needs a whole session.

## 6 · Online — evaluating a trace with no answer key

Everything above had a dataset. Production does not. What survives?

In [ ]:
# Runs the healthy agent, then scores its TRACE — not its outputs.
from datetime import datetime, timedelta, timezone
since = datetime.now(timezone.utc) - timedelta(seconds=5)

evalkit.build_agent().invoke({"messages": [INPUTS["question"]]})
evalkit.flush_traces()          # gotcha #6: traces send on a background thread

root_id = evalkit.find_trace_root(client, since=since)
root, note = evalkit.read_run_when_ready(client, root_id)   # gotcha #16
print("note:", note or "(clean)")
print("outermost tool spans:", len(evalkit.walk_tool_spans(root)))  # gotcha #8

for ev in evalkit.ONLINE_EVALUATORS:
    print(" ", ev(root))

`walk_tool_spans` counts **outermost** spans only. LangGraph emits two spans per
tool call — the tools-node span, and the `@tool` function's own span nested inside
it, same name, same arguments. Count both and every single-search run looks like it
issued a duplicate query, and your healthy control gets classified as broken.

**That is the third harness bug in this notebook.** When a measurement surprises you,
suspect the thing doing the measuring first.

---
## Where this leaves you

You built four things: a dataset row, an evaluator, a matrix that proves the
evaluator can fail, and an experiment.

The one that matters is the third. Anyone can write a check. **The discipline is
refusing to trust one that has never failed.**

| | |
|---|---|
| **Session 5** | How many rows does a dataset need? Adversarial and edge-case benchmarks. |
| **Session 6** | Regression testing — is version N+1 worse than version N? |
| **Session 8** | Is your judge any good? Rubrics and bias. |
| **Session 11** | State evaluation, on a workflow that has state worth checking. |

### Homework
1. Your written judgement about the non-discriminating evaluator — the field above.
2. Add your validated dataset row to the shared dataset.
3. `my_evaluators.py`, committed, containing one evaluator of your own that
   passes `healthy` and fails at least one broken seed — plus the
   `test_my_evaluators.py` output pasted in.